# Single-stream masked JEPA prototype

This notebook is a **small Health&Gait feasibility experiment**. It tests whether a single video stream can learn non-collapsed representations by predicting masked target-encoder features. It intentionally follows the masked V-JEPA training pattern: the context and target encoders see complementary token regions from the same clip.

It is **not** the final CoDy-JEPA model and does not implement counterfactual token swapping, factorized streams, or explicit future-dynamics prediction. Success requires more than falling loss: validation representations must retain variance/effective rank, and shuffling video context must make prediction worse.

`Run All` is safe: it validates data and runs a tiny CPU smoke test. A full CUDA job is enabled with `CODY_JEPA_RUN_FULL_TRAINING=1`; exhaustive frame certification is a separate, opt-in CPU/I/O job so it cannot silently consume the GPU allocation before training. The production train-to-report workflow is `uv run python scripts/run_phase0_pipeline.py`; it invokes this notebook inside the local GPU or Slurm allocation, then validates the checkpoint, exports features, runs probes, and writes a report. Launch this notebook with `uv run jupyter lab notebooks/single-stream-jepa.ipynb`; manage the locked environment only with `uv sync`, `uv add`, `uv remove`, and `uv lock`, and launch every Python or Jupyter command through `uv run`.

In [ ]:
from pathlib import Path
import json
import math
import os
import random
import sys
import time

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import torch
from torch.utils.data import DataLoader

NOTEBOOK_CWD = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_CWD if (NOTEBOOK_CWD / 'src').exists() else NOTEBOOK_CWD.parent
for import_root in (PROJECT_ROOT, PROJECT_ROOT / 'src'):
    if str(import_root) not in sys.path:
        sys.path.insert(0, str(import_root))

from cody_jepa.data import (
    HealthGaitLoaderConfig,
    audit_healthgait_clip_quality,
    build_healthgait_datasets_from_config,
    build_healthgait_loaders_from_config,
    find_repo_root,
    healthgait_manifest_path,
)
from cody_jepa.single_stream_jepa import (
    DEFAULT_MASK_GROUPS,
    MODEL_ARCHITECTURE,
    build_models,
    healthy_checkpoint_path,
    load_checkpoint,
    multiblock_mask,
    resolve_device,
    train_jepa,
    video_from_batch,
)

def print_json(payload):
    print(json.dumps(payload, indent=2, sort_keys=True))

def env_flag(name, default=False):
    value = os.environ.get(name)
    if value is None:
        return bool(default)
    normalized = value.strip().casefold()
    if normalized not in {'0', '1', 'false', 'true', 'no', 'yes'}:
        raise ValueError(f'{name} must be a boolean flag; got {value!r}')
    return normalized in {'1', 'true', 'yes'}

RUN_FULL_TRAINING = env_flag('CODY_JEPA_RUN_FULL_TRAINING', False)
RUN_DATA_AUDIT = env_flag('CODY_JEPA_RUN_DATA_AUDIT', not RUN_FULL_TRAINING)
RUN_EXHAUSTIVE_DATA_AUDIT = env_flag(
    'CODY_JEPA_RUN_EXHAUSTIVE_DATA_AUDIT', False
)

CONFIG = {
    'seed': 0,
    # batch_size is a physical microbatch; four microbatches make an effective 64.
    'batch_size': 16,
    'accumulation_steps': 4,
    'steps': 3_900,
    'num_epochs': 100,
    'lr': 1e-4,
    'start_lr': 1e-5,
    'warmup_steps': 200,
    'min_lr': 1e-6,
    'weight_decay': 0.04,
    'grad_clip': 1.0,
    'ema_start': 0.99,  # softer early EMA so the target does not rigidly track the online net into the shortcut basin
    'ema_end': 1.0,
    'loss_exp': 1.0,
    # VICReg-style anti-collapse safeguards (README section 4):
    # variance hinge keeps every feature dimension alive and
    # covariance decorrelation prevents the rank-2 plateau seen
    # in HAIC runs 90881/91023 with EMA-only anti-collapse.
    'var_coef': 1.0,
    'cov_coef': 0.04,
    'var_gamma': 1.0,
    # Standardize target dims across the batch to remove the
    # constant-prediction escape hatch behind the step<780 collapse.
    'target_batch_standardize': True,

    'num_frames': 16,
    'img_size': 112,
    'patch_size': 8,
    'tubelet_size': 2,
    'in_channels': 1,
    'num_tokens': 1568,
    'min_context_tokens': 64,
    'input_mean': 0.5,
    'input_std': 0.5,

    'embed_dim': 384,
    'hidden_dim': 1536,
    'num_heads': 6,
    'num_layers': 6,
    'pred_dim': 192,
    'pred_depth': 6,
    'dropout': 0.0,
    'uniform_power': True,
    'norm_eps': 1e-6,

    'amp_dtype': 'bfloat16',
    'compile': False,  # enable only after the eager CUDA preflight succeeds
    'tf32': True,
    'required_device': 'cuda',
    'eval_every_epochs': 5,
    'train_eval_every_epochs': 0,
    'checkpoint_every_epochs': 1,
    'selection_metric': 'subject_balanced_loss',
    'min_feature_std': 1e-3,
    'max_near_zero_variance_fraction': 0.5,
    'min_effective_rank_ratio': 0.05,
    'min_context_shuffle_loss_gap': 1e-3,
}

assert CONFIG['num_tokens'] == (
    CONFIG['num_frames'] // CONFIG['tubelet_size']
    * (CONFIG['img_size'] // CONFIG['patch_size']) ** 2
)
cuda_preflight = None
if RUN_FULL_TRAINING:
    FULL_TRAINING_DEVICE = resolve_device(CONFIG['required_device'])
    cuda_preflight = {
        'status': 'passed',
        'cuda_device_name': torch.cuda.get_device_name(FULL_TRAINING_DEVICE),
    }
print_json({
    'architecture': MODEL_ARCHITECTURE,
    'effective_batch': CONFIG['batch_size'] * CONFIG['accumulation_steps'],
    'run_full_training': RUN_FULL_TRAINING,
    'run_data_audit': RUN_DATA_AUDIT,
    'run_exhaustive_data_audit': RUN_EXHAUSTIVE_DATA_AUDIT,
    'cuda_preflight': cuda_preflight,
})

In [ ]:
REPO_ROOT = find_repo_root()
MANIFEST_CSV = healthgait_manifest_path(REPO_ROOT)
LOADER_CONFIG = HealthGaitLoaderConfig(
    manifest_csv=MANIFEST_CSV,
    repo_root=REPO_ROOT,
    split='train',
    clip_length=CONFIG['num_frames'],
    image_size=(CONFIG['img_size'], CONFIG['img_size']),
    batch_size=CONFIG['batch_size'],
    seed=CONFIG['seed'],
    num_workers=4,
    pin_memory=torch.cuda.is_available(),
    prefetch_factor=1,
    train_crop_scale=(0.90, 1.0),
    train_horizontal_flip_prob=0.5,
    strict_frame_sequence=True,
    image_verify_mode='all' if RUN_EXHAUSTIVE_DATA_AUDIT else 'sample',
    inventory_hash_mode='full' if RUN_EXHAUSTIVE_DATA_AUDIT else 'sample',
    allowed_data_root=REPO_ROOT / 'data' / 'healthgait',
    eval_windows=3,
    drop_last_train=True,
)

# Build each dataset once; the validation dataset reuses the validated manifest inventory.
train_ds, val_ds = build_healthgait_datasets_from_config(LOADER_CONFIG)
train_loader, val_loader = build_healthgait_loaders_from_config(
    LOADER_CONFIG, datasets=(train_ds, val_ds)
)
portable_loader_config = {
    key: value
    for key, value in LOADER_CONFIG.as_dict().items()
    if key not in {'manifest_csv', 'repo_root', 'allowed_data_root'}
}
DATA_CONTRACT = {
    'loader_config': portable_loader_config,
    'train_dataset': train_ds.signature(),
    'val_dataset': val_ds.signature(),
}
print_json({
    'train_sequences': len(train_ds.samples),
    'train_clips_per_epoch': len(train_ds),
    'val_sequences': len(val_ds.samples),
    'val_deterministic_windows': len(val_ds),
    'data_contract': DATA_CONTRACT,
})

In [ ]:
# The all-sequence quality audit is intentionally separate from full training.
# On HAIC it is useful before submission but wastes an allocated H100 if repeated.
quality_summary = (
    audit_healthgait_clip_quality((train_ds, val_ds)) if RUN_DATA_AUDIT else None
)
preflight_loader = DataLoader(train_ds, batch_size=4, shuffle=False, num_workers=0)
preflight_batch = next(iter(preflight_loader))
preflight_video = video_from_batch(
    preflight_batch, torch.device('cpu'), CONFIG, expected_split='train'
)
train_subjects = {sample['subject_id'].casefold() for sample in train_ds.samples}
val_subjects = {sample['subject_id'].casefold() for sample in val_ds.samples}
assert train_subjects.isdisjoint(val_subjects)
assert torch.isfinite(preflight_video).all()
print_json({
    'batch_shape': tuple(preflight_batch['video'].shape),
    'pixel_range': (
        float(preflight_batch['video'].min()),
        float(preflight_batch['video'].max()),
    ),
    'quality_summary': quality_summary,
    'train_subjects': len(train_subjects),
    'val_subjects': len(val_subjects),
})

In [ ]:
def smoke_test_training_loop():
    smoke_cfg = {
        **CONFIG,
        'batch_size': 2, 'accumulation_steps': 1, 'steps': 1, 'num_epochs': 1,
        'lr': 1e-3, 'start_lr': 1e-3, 'warmup_steps': 1, 'min_lr': 1e-5,
        'ema_start': 0.9, 'ema_end': 0.9,
        'num_frames': 4, 'img_size': 16, 'patch_size': 4, 'tubelet_size': 2,
        'num_tokens': 32, 'min_context_tokens': 4,
        'embed_dim': 12, 'hidden_dim': 24, 'num_heads': 3, 'num_layers': 1,
        'pred_dim': 12, 'pred_depth': 1,
        'amp_dtype': None, 'compile': False, 'tf32': False,
        'required_device': 'cpu', 'eval_every_epochs': 1,
    }
    generator = torch.Generator().manual_seed(smoke_cfg['seed'])
    train_samples = [{
        'video': torch.rand(4, 1, 16, 16, generator=generator),
        'split': 'train', 'sequence_id': f'train-{index}', 'subject_id': f'T{index}',
    } for index in range(2)]
    val_samples = [{
        'video': torch.rand(4, 1, 16, 16, generator=generator),
        'split': 'val', 'sequence_id': f'val-{index}', 'subject_id': f'V{index}',
    } for index in range(2)]
    smoke_train = DataLoader(
        train_samples, batch_size=2, shuffle=True,
        generator=torch.Generator().manual_seed(smoke_cfg['seed']),
    )
    smoke_val = DataLoader(val_samples, batch_size=2, shuffle=False)
    result = train_jepa(
        smoke_cfg, smoke_train, smoke_val, {'dataset': 'synthetic-smoke-v1'},
        checkpoint_dir=None, device='cpu',
    )
    assert result['global_step'] == 1
    assert result['completed_epochs'] == 1
    assert result['history'][0]['val']['effective_rank'] > 0
    assert not any(p.requires_grad for p in result['target_encoder'].parameters())
    assert all(p.grad is None for p in result['target_encoder'].parameters())
    masks_a = multiblock_mask(smoke_cfg, 2, random.Random(99))
    masks_b = multiblock_mask(smoke_cfg, 2, random.Random(99))
    assert all(torch.equal(a['ctx'], b['ctx']) for a, b in zip(masks_a, masks_b))
    return result['history'][0]

smoke_metrics = smoke_test_training_loop()
print_json({'smoke_metrics': smoke_metrics})

In [ ]:
# CPU-only structural preflight. The full CUDA run still performs the real BF16
# forward/backward path and fails fast if the selected GPU is unsuitable.
production_masks = multiblock_mask(CONFIG, 2, random.Random(CONFIG['seed']))
for group in production_masks:
    assert group['ctx'].shape[0] == 2 and group['pred'].shape[0] == 2
    assert not torch.isin(group['ctx'][0], group['pred'][0]).any()
production_context, production_target, production_predictor = build_models(
    CONFIG, torch.device('cpu')
)
parameter_counts = {
    'context_encoder': sum(p.numel() for p in production_context.parameters()),
    'target_encoder': sum(p.numel() for p in production_target.parameters()),
    'predictor': sum(p.numel() for p in production_predictor.parameters()),
}
assert production_context.num_patches == CONFIG['num_tokens']
del production_context, production_target, production_predictor
print_json({
    'parameter_counts': parameter_counts,
    'mask_shapes': [(g['label'], tuple(g['ctx'].shape), tuple(g['pred'].shape)) for g in production_masks],
    'input_microbatch_mib': (
        CONFIG['batch_size'] * CONFIG['num_frames'] * CONFIG['img_size'] ** 2 * 4 / 2**20
    ),
})

In [ ]:
output_dir_value = os.environ.get(
    'CODY_JEPA_OUTPUT_DIR', 'outputs/jepa-v5'
)
OUTPUT_DIR = Path(output_dir_value).expanduser()
if not OUTPUT_DIR.is_absolute():
    OUTPUT_DIR = PROJECT_ROOT / OUTPUT_DIR
resume_value = os.environ.get('CODY_JEPA_RESUME_CHECKPOINT')
RESUME_CHECKPOINT = Path(resume_value).expanduser() if resume_value else None
if RESUME_CHECKPOINT is not None and not RESUME_CHECKPOINT.is_absolute():
    RESUME_CHECKPOINT = PROJECT_ROOT / RESUME_CHECKPOINT

result = None
if RUN_FULL_TRAINING:
    if not torch.cuda.is_available():
        raise RuntimeError('full training requires an allocated CUDA GPU')
    if RESUME_CHECKPOINT is None and (OUTPUT_DIR / 'latest.pt').exists():
        raise FileExistsError(
            f'{OUTPUT_DIR / "latest.pt"} already exists; choose a new OUTPUT_DIR or resume it'
        )
    resume_state = (
        load_checkpoint(RESUME_CHECKPOINT) if RESUME_CHECKPOINT is not None else None
    )

    # Exercise the production BF16 forward/backward path with one real physical
    # microbatch. This reaches the H100 quickly and fails before the long run on
    # incompatible kernels, OOM, non-finite gradients, or broken data transfer.
    preflight_train_batch = next(iter(train_loader))
    preflight_val_batch = next(iter(val_loader))
    preflight_cfg = {
        **CONFIG,
        'accumulation_steps': 1, 'steps': 1, 'num_epochs': 1,
        'warmup_steps': 1, 'eval_every_epochs': 1,
    }
    preflight_started = time.perf_counter()
    preflight_result = train_jepa(
        preflight_cfg,
        [preflight_train_batch],
        [preflight_val_batch],
        {'dataset': 'real-batch-cuda-preflight-v1'},
        checkpoint_dir=None,
        device=FULL_TRAINING_DEVICE,
    )
    torch.cuda.synchronize(FULL_TRAINING_DEVICE)
    cuda_training_preflight = {
        'status': 'passed',
        'seconds': time.perf_counter() - preflight_started,
        'batch_size': int(preflight_train_batch['video'].shape[0]),
        'peak_gpu_memory_mib': preflight_result['history'][0]['peak_gpu_memory_mib'],
        'loss': preflight_result['history'][0]['train_loss'],
    }
    del preflight_result, preflight_train_batch, preflight_val_batch
    torch.cuda.empty_cache()
    print_json({'cuda_training_preflight': cuda_training_preflight})
    print_json({'training_status': 'starting', 'output_dir': str(OUTPUT_DIR)})
    result = train_jepa(
        CONFIG,
        train_loader,
        val_loader,
        DATA_CONTRACT,
        checkpoint_dir=OUTPUT_DIR,
        resume_state=resume_state,
    )
    best_healthy_path = healthy_checkpoint_path(
        OUTPUT_DIR, result['best_healthy_epoch']
    )
    print_json({
        'latest': str(OUTPUT_DIR / 'latest.pt'),
        'best_loss': str(OUTPUT_DIR / 'best_loss.pt'),
        'best_healthy': str(best_healthy_path) if best_healthy_path else None,
        'best_epoch': result['best_epoch'],
        'best_val_loss': result['best_val_loss'],
        'best_healthy_epoch': result['best_healthy_epoch'],
        'termination_reason': result['termination_reason'],
    })
else:
    print_json({
        'full_training': 'skipped',
        'next_step': 'Set CODY_JEPA_RUN_FULL_TRAINING=1 on a CUDA worker.',
    })

In [ ]:
if result is None:
    print_json({'plot_status': 'skipped', 'reason': 'no_full_run_result'})
else:
    history = result['history']
    steps = [row['step'] for row in history]
    train_loss = [row['train_loss'] for row in history]
    train_cosine = [row['train_cosine'] for row in history]
    eval_rows = [row for row in history if row['val'] is not None]
    eval_steps = [row['step'] for row in eval_rows]
    val_loss = [row['val']['loss'] for row in eval_rows]
    val_cosine = [row['val']['cosine'] for row in eval_rows]
    effective_rank = [row['val']['effective_rank'] for row in eval_rows]
    feature_std = [row['val']['feature_std'] for row in eval_rows]
    near_zero_fraction = [row['val']['near_zero_variance_fraction'] for row in eval_rows]
    shuffle_gap = [
        row['val']['subject_balanced_context_shuffle_loss_gap']
        for row in eval_rows
    ]

    fig, axes = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)
    axes[0, 0].plot(steps, train_loss, label='train')
    axes[0, 0].plot(eval_steps, val_loss, 'o-', label='validation')
    axes[0, 0].set_title('JEPA loss')
    axes[0, 1].plot(steps, train_cosine, label='train')
    axes[0, 1].plot(eval_steps, val_cosine, 'o-', label='validation')
    axes[0, 1].set_title('Cosine similarity')
    axes[1, 0].plot(eval_steps, effective_rank, 'o-')
    axes[1, 0].set_title('Online full-view effective rank')
    axes[0, 2].plot(eval_steps, feature_std, 'o-')
    axes[0, 2].set_title('Online full-view feature std')
    axes[1, 1].plot(eval_steps, near_zero_fraction, 'o-')
    axes[1, 1].set_title('Online full-view near-zero variance fraction')
    axes[1, 2].plot(eval_steps, shuffle_gap, 'o-')
    axes[1, 2].axhline(0, color='black', linewidth=1)
    axes[1, 2].set_title('Subject-balanced wrong-context loss gap')
    for axis in axes.flat:
        axis.set_xlabel('optimizer step')
        axis.xaxis.set_major_locator(MaxNLocator(8, integer=True))
        axis.grid(True, alpha=0.3)
        if axis.get_legend_handles_labels()[0]:
            axis.legend(frameon=False)
    plt.show()